In [ ]:
import pandas as pd

# =====================================================
# 1. 기존 BERTopic 결과 불러오기
# =====================================================

INPUT_FILE = "tottenham_cafe_bertopic_result.xlsx"
OUTPUT_FILE = "tottenham_cafe_bertopic_macro_result.xlsx"

# 저장할 때 sheet_name을 다르게 했을 수 있으니 우선 시트 확인
xls = pd.ExcelFile(INPUT_FILE)
print(xls.sheet_names)

# 보통 topic_labeled_data 또는 cafe_topic_labeled_data에 원본+topic 결과가 있음
SHEET_NAME = "cafe_topic_labeled_data" if "cafe_topic_labeled_data" in xls.sheet_names else xls.sheet_names[0]

topic_df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

print(topic_df.shape)
print(topic_df.columns.tolist())

display(topic_df.head())

In [ ]:
# # =====================================================
# # 2. topic_name 없으면 생성
# # =====================================================

# if "topic_name" not in topic_df.columns:
#     if "Name" in topic_df.columns:
#         topic_df["topic_name"] = topic_df["Name"]
#     else:
#         topic_df["topic_name"] = topic_df["topic"].astype(str)

# print(topic_df[["topic", "topic_name"]].drop_duplicates().head(30))

In [ ]:
# =====================================================
# topic_info에서 Representation 붙이기
# =====================================================

topic_info = pd.read_excel(INPUT_FILE, sheet_name="topic_info")

rep_map = topic_info.set_index("Topic")["Representation"].to_dict()

topic_df["Representation"] = topic_df["topic"].map(rep_map)

display(
    topic_df[["topic", "topic_name", "Representation"]]
    .drop_duplicates()
    .sort_values("topic")
    .head(30)
)

In [ ]:
# =====================================================
# 세부 토픽 -> 상위 토픽 매핑: 키워드 점수 기반
# =====================================================

macro_keywords = {
    "이적시장/선수보강": [
        "이적", "시장", "영입", "임대", "계약", "오피셜", "주급", "가격",
        "이적료", "디렉터", "레비", "랑게", "비나이", "오시멘", "레앙",
        "은디아예", "스콧", "기라시", "쿠두스", "오도베르", "사비뉴",
        "요렌테", "팔리냐", "갤러거", "무아니", "래시포드", "유망주",
        "매물", "방출", "판매", "보강", "스트라이커", "공미", "공격수",
        "윙어", "티아고", "요케레스", "누사", "히샬리송"
    ],

    "잔류/강등/시즌 결과": [
        "잔류", "강등", "승점", "순위", "마지막", "확정", "원정",
        "더블", "coys", "고생", "마음", "승리", "대항전", "기록",
        "패배", "우승", "경쟁", "중위"
    ],

    "감독/전술 방향성": [
        "데제르비", "투도르", "프랭크", "전술", "압박", "연계",
        "빌드업", "라인업", "교체", "성향", "선임", "감독"
    ],

    "선수 기량/포지션 평가": [
        # 플레이/포지션/능력 관련 일반 키워드
        "플레이", "능력", "왼발", "오른발", "속도", "피지컬", "민첩성",
        "헤더", "공격", "수비", "가담", "압박", "패스", "득점력",
        "포지션", "미드필더", "공미", "윙어", "스트라이커", "공격수",
        "풀백", "센터백", "키퍼", "골키퍼", "주전", "백업",
        "프로", "비판", "태도", "실수", "단점", "장점",

        # 기존에 있던 핵심 선수/표현
        "손흥민", "시몬스", "사비시몬스", "그레이", "텔",
        "히샬리송", "히샬", "히살리송",
        "로메로", "포로", "판더펜", "판더벤", "반더벤",
        "베리발", "비수마", "벤탕쿠르",

        # 감정분석 프롬프트에 있던 토트넘 선수단
        "비카리오", "굴리엘모", "드라구신", "라구드라구신",
        "단소", "케빈단소",
        "제임스매디슨", "매디슨",
        "마티스텔", "데스티니우도기", "우도기",
        "아치그레이", "루카스베리발",
        "크리스티안로메로", "도미닉솔랑케", "솔랑케",
        "모하메드쿠드스", "쿠드스", "쿠두스",
        "데얀클루셉스키", "클루셉스키", "클루셉", "셉셉",
        "코너갤러거", "갤러거",
        "페드로포로", "제드스펜스", "스펜스",
        "윌손오도베르", "오도베르",
        "파페마타르사르", "사르",
        "로드리고벤탕쿠르",
        "안토닌킨스키", "킨스키",
        "벤데이비스", "데이비스",
        "미키판더펜",
        "소우자",
        "랑달콜로무아니", "콜로무아니", "무아니",
        "브랜던오스틴", "오스틴"
    ],

    "구단 운영/보드진 비판": [
        "보드진", "수뇌부", "구단주", "무능", "운영",
        "실망", "최악", "위험", "충격", "레비", "랑게", "비나이"
    ],

    "부상/복귀 이슈": [
        "부상", "회복", "복귀", "의료진", "메디컬", "클루셉스키",
        "장기", "관리"
    ],

    "골키퍼/수비진 평가": [
        "킨스키", "키퍼", "골키퍼", "선방", "주전", "비카리오",
        "페르브뤼헌", "트래포드", "달로우", "수비진"
    ],

    "팬 반응/소통": [
        "감사", "덕분", "의견", "설명", "공감", "동의",
        "동감", "행복", "나이스", "수고"
    ]
}


def assign_macro_topic_score(topic_text):
    t = str(topic_text).lower()


    scores = {}

    for macro, keywords in macro_keywords.items():
        matched = [k for k in keywords if k in t]
        scores[macro] = len(matched)

    # 가장 많이 매칭된 카테고리 선택
    best_macro = max(scores, key=scores.get)
    best_score = scores[best_macro]

    # 아무 키워드도 안 걸리면 기타
    if best_score == 0:
        return "기타/분류불가"

    return best_macro

In [ ]:
topic_df["macro_topic"] = topic_df["topic_name"].apply(assign_macro_topic_score)

display(
    topic_df[["topic", "topic_name", "macro_topic"]]
    .drop_duplicates()
    .sort_values("topic")
)

In [ ]:
def get_macro_scores(topic_text):
    t = str(topic_text).lower()

    scores = {}

    for macro, keywords in macro_keywords.items():
        matched = [k for k in keywords if k in t]
        scores[macro] = {
            "score": len(matched),
            "matched_keywords": matched
        }

    return scores


# topic별 점수 확인용 테이블 만들기
score_rows = []

for _, row in (
    topic_df[["topic", "Representation"]]
    .drop_duplicates()
    .sort_values("topic")
    .iterrows()
):
    if row["topic"] == -1:
        macro_topic = "기타/분류불가"
        scores = {}
    else:
        scores = get_macro_scores(row["Representation"])
        macro_topic = assign_macro_topic_score(row["Representation"])

    score_rows.append({
        "topic": row["topic"],
        "Representation": row["Representation"],
        "macro_topic": assign_macro_topic_score(row["Representation"]),
        "scores": scores
    })

score_df = pd.DataFrame(score_rows)

display(score_df)

In [ ]:
# # =====================================================
# # 5. 필요하면 topic 번호 기준 수동 보정
# # =====================================================

# manual_override = {
#     # 예시: topic 번호: "상위 토픽명"
#     # 7: "선수 기량/포지션 평가",
#     # 8: "이적시장/선수보강",
#     # 10: "이적시장/선수보강",
#     # 11: "이적시장/선수보강",
#     # 18: "이적시장/선수보강",
#     # 22: "이적시장/선수보강",
# }

# for topic_id, macro in manual_override.items():
#     topic_df.loc[topic_df["topic"] == topic_id, "macro_topic"] = macro

# display(
#     topic_df[["topic", "topic_name", "macro_topic"]]
#     .drop_duplicates()
#     .sort_values("topic")
# )

In [ ]:
# =====================================================
# 6. 38R 전후 상위 토픽 비교
# =====================================================

period_count = pd.crosstab(
    topic_df["topic_period"],
    topic_df["macro_topic"]
)

period_ratio = pd.crosstab(
    topic_df["topic_period"],
    topic_df["macro_topic"],
    normalize="index"
) * 100

display(period_count)
display(period_ratio.round(2))

In [ ]:
# =====================================================
# 7. 긍정/부정별 상위 토픽 비교
# =====================================================

pn_df = topic_df[topic_df["gpt_sentiment"].isin(["positive", "negative"])].copy()

sentiment_count = pd.crosstab(
    pn_df["gpt_sentiment"],
    pn_df["macro_topic"]
)

sentiment_ratio = pd.crosstab(
    pn_df["gpt_sentiment"],
    pn_df["macro_topic"],
    normalize="index"
) * 100

display(sentiment_count)
display(sentiment_ratio.round(2))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# =====================================================
# 한글 폰트 설정
# =====================================================

def set_korean_font():
    font_candidates = [
        "Malgun Gothic",      # Windows
        "AppleGothic",        # Mac
        "NanumGothic",        # Linux / Colab
        "Noto Sans CJK KR"
    ]

    installed_fonts = {f.name: f.fname for f in fm.fontManager.ttflist}

    for font in font_candidates:
        if font in installed_fonts:
            plt.rcParams["font.family"] = font
            plt.rcParams["axes.unicode_minus"] = False
            print(f"✅ 한글 폰트 설정 완료: {font}")
            return installed_fonts[font]

    print("⚠️ 한글 폰트를 찾지 못했습니다. Windows면 아래 직접 지정 코드를 사용하세요.")
    return None


font_path = set_korean_font()

In [ ]:
import re
from kiwipiepy import Kiwi

kiwi = Kiwi()

custom_words = [
    "래시포드", "사비뉴", "오도베르", "데제르비", "데버지",
    "히샬리송", "클루셉스키", "비카리오", "킨스키",
    "로메로", "판더펜", "우도기", "포로", "매디슨",
    "쿠두스", "솔랑케", "베리발", "드라구신",
    "토트넘조롱"
]

for word in custom_words:
    kiwi.add_user_word(word, "NNP")


stopwords = set([
    # [1] 기존 & 유튜브/커뮤니티 은어
    '진짜', '너무', '영상', '오늘', '경기', '그냥', '진심', 'ㅋㅋ', 'ㅎㅎ', 'ㅠㅠ', 'ㅜㅜ', 
    '이거', '저거', '그거', '뭔가', '정말', '지금', '다시', '계속', '가장', '제일', '조금', '정도','개추',
    
    # [2] 축구 일반 용어 (분석 인사이트가 없는 뻔한 단어)
    '축구', '하이라이트', '감독', '선수', '시즌', '리그', '전반', '후반', '골', '볼', '팬', '응원', '생각', '사람', '우리', '이번',
    
    # [3] 토트넘 지칭
    '토트넘', '스퍼스', '닭집',
    
    # [4] 타 팀 및 관련 선수 (노이즈 제거)
    '맨유', '맨체스터유나이티드', '맹구', '맨시티', '맨체스터시티', 
    '아스날', '아스널', '개스널', '리버풀', '전범풀', '첼시', '첼강딱', 
    '뉴캐슬', '빌라', '에버튼', '울버햄튼', '황희찬', 
    '레알', '바르샤', '뮌헨', '김민재', '이강인', '파리', 'psg', '리즈',
    '웨스트햄', '아스톤빌라','웨햄','클럽',
    
    
    # [6] 형태소 분석(Okt) 후 흔히 남는 무의미한 두 글자 명사들
    '때문', '위해', '대한', '관련', '진짜' ,"작성","짧은","글", "질문", "삭제", "필수", "검색",
    "경기시작", "익일", "오전" ,"일반","일반글", "작성", "금지","게시","필독","성의","숙지","상단","등업",
    "카페", "댓글", "게시글",
    "팬", "회원", "작성", "게시", "필수", "일반", "삭제", "익일", "필독",
    "공지", "규정", "등업", "글쓰기", "제목"
])

stopwords = set(str(w).strip().lower() for w in stopwords)

def kiwi_tokenizer(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []

    for token in kiwi.tokenize(text):
        if token.tag in ["NNG", "NNP", "SL"]:
            word = token.form.strip().lower()

            if len(word) >= 2 and word not in stopwords:
                tokens.append(word)

    return tokens

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd

# 색상
TOTAL_COLOR = "#D8E3F0"   # 전체 데이터: 연한 색
TOPIC_COLOR = "#132257"   # 특정 토픽: 진한 색

TEXT_COL = "topic_text" if "topic_text" in topic_df.columns else "text_clean"


def count_keywords(data):
    counter = Counter()

    for text in data[TEXT_COL].dropna().astype(str):
        counter.update(kiwi_tokenizer(text))

    return counter


# 전체 데이터 키워드 카운트
total_counter = count_keywords(topic_df)


def plot_topic_keywords(macro_topic, top_n=15):
    topic_data = topic_df[topic_df["macro_topic"] == macro_topic].copy()
    topic_counter = count_keywords(topic_data)

    top_words = [w for w, c in topic_counter.most_common(top_n)]

    plot_df = pd.DataFrame({
        "keyword": top_words,
        "topic_count": [topic_counter[w] for w in top_words],
        "total_count": [total_counter[w] for w in top_words]
    })

    x = range(len(plot_df))

    plt.figure(figsize=(12, 5))

    plt.bar(
        x,
        plot_df["total_count"],
        color=TOTAL_COLOR,
        label="전체 데이터 키워드 수"
    )

    plt.bar(
        x,
        plot_df["topic_count"],
        color=TOPIC_COLOR,
        label=f"{macro_topic} 키워드 수"
    )

    plt.xticks(x, plot_df["keyword"], rotation=45, ha="right")
    plt.title(f"{macro_topic} 주요 키워드 비교", fontsize=15, fontweight="bold")
    plt.ylabel("키워드 등장 횟수")
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()


# 가장 많은 상위 토픽 3개
top3_macros = topic_df["macro_topic"].value_counts().head(3).index.tolist()

for macro in top3_macros:
    plot_topic_keywords(macro, top_n=15)

In [ ]:
# 보고 싶은 상위 토픽명 직접 지정
TARGET_MACRO_TOPIC = "이적시장/선수보강"

plot_topic_keywords(
    TARGET_MACRO_TOPIC,
    top_n=15
)

In [ ]:
# =====================================================
# 8. 저장
# =====================================================

topic_macro_check = (
    topic_df
    .groupby(["topic", "topic_name", "macro_topic"])
    .size()
    .reset_index(name="count")
    .sort_values("topic")
)

with pd.ExcelWriter(OUTPUT_FILE) as writer:
    topic_df.to_excel(writer, sheet_name="topic_macro_data", index=False)
    topic_macro_check.to_excel(writer, sheet_name="topic_macro_check", index=False)
    period_count.to_excel(writer, sheet_name="period_macro_count")
    period_ratio.round(2).to_excel(writer, sheet_name="period_macro_ratio")
    sentiment_count.to_excel(writer, sheet_name="sentiment_macro_count")
    sentiment_ratio.round(2).to_excel(writer, sheet_name="sentiment_macro_ratio")

print("저장 완료:", OUTPUT_FILE)